In [1]:
from observix.evaluation import CustomEvaluator

[OBSERVIX] Initialized with Host: http://localhost:8000


In [ ]:
CustomEvaluator()

In [ ]:
import asyncio
import json
import os
from dotenv import load_dotenv

from observix.evaluation.integrations.observix_eval import CustomEvaluator
from observix.schema import Trace, Observation

load_dotenv()

async def main():
    print("Setting up CustomEvaluator...")
    
    # We will use langchain provider (Groq) since we have a key in comprehensive_eval.py normally,
    # or openai if key exists. Let's just use whatever is configured there.
    # checking env
    if not os.getenv("OPENAI_API_KEY") and not os.getenv("GROQ_API_KEY"):
        print("Please set OPENAI_API_KEY or GROQ_API_KEY in the environment or .env file.")
        return

    provider = "langchain" if os.getenv("OPENAI_API_KEY") else "langchain"
    model = "openai/gpt-oss-120b" if provider == "openai" else "openai/gpt-oss-120b"
    
    evaluator = CustomEvaluator(
        provider=provider,
        model=model,
    )
    
    print(f"Evaluator created with provider={provider}, model={model}")
    
    trace_dict = {
        "id": 100,
        "trace_id": "test_custom",
        "name": "TestTrace",
        "duration_ms": 1000.0,
        "observations": [
            {
                "id": 1,
                "type": "chain",
                "name": "workflow_root",
                "start_time": 1000,
                "end_time": 2000,
                "input_text": '{"query": "Is it sunny?"}',
                "output_text": "Yes, it's sunny."
            }
        ]
    }
    
    custom_instructions = "The output must contain the word 'sunny'."
    
    print("Evaluating...")
    try:
        result = await evaluator.evaluate(
            trace={"trace": trace_dict, "context": {"agents": [], "tools": []}},
            criteria=custom_instructions
        )
        print("\n=== Eval Result ===")
        print(f"Passed: {result.passed}")
        print(f"Score: {result.score}")
        print(f"Reason: {result.reason}")
        print("Metadata:", json.dumps(result.metadata, indent=2))
    except Exception as e:
        print(f"Evaluation failed: {e}")

print(await main()
)

Setting up CustomEvaluator...
Evaluator created with provider=langchain, model=openai/gpt-oss-120b
Evaluating...

=== Eval Result ===
Passed: True
Score: 1.0
Reason: The only custom criterion requires that the workflow's final output contain the word 'sunny'. The trace shows a single observation where the output_text is "Yes, it's sunny.", which clearly includes the required word. No other criteria are defined, and there are no agents or tools involved that could affect compliance. Therefore the workflow fully satisfies the custom requirement.
Metadata: {
  "full_evaluation_details": {
    "score": 100,
    "reasoning": "The only custom criterion requires that the workflow's final output contain the word 'sunny'. The trace shows a single observation where the output_text is \"Yes, it's sunny.\", which clearly includes the required word. No other criteria are defined, and there are no agents or tools involved that could affect compliance. Therefore the workflow fully satisfies the custo

Failed to export observation to backend: Server error '500 Internal Server Error' for url 'http://localhost:8000/api/v1/ingest/observations'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
Failed to export traces to backend (http://localhost:8000): Server error '500 Internal Server Error' for url 'http://localhost:8000/api/v1/ingest/traces'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
